In [23]:
import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_curve, auc

In [17]:
mlflow.set_experiment("Potabilidad_Arbol_Experimento")

<Experiment: artifact_location='/home/ygalindo/proyectoDesarrollo/WaterPotability/notebooks/mlruns/1', creation_time=1771704407346, experiment_id='1', last_update_time=1771704407346, lifecycle_stage='active', name='Potabilidad_Arbol_Experimento', tags={}, workspace='default'>

In [18]:
data = pd.read_csv("../data/Datos_Etapa-2.csv", sep=",", na_values=["?"])

data["Potabilidad"] = data["Potabilidad"].map({"NO": 0, "SI": 1})

X = data.drop("Potabilidad", axis=1)
y = data["Potabilidad"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
)

In [20]:
numeric_features = ['pH', 'Sulfatos', 'Trihalometanos']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        class_weight='balanced',
        random_state=77
    ))
])

In [21]:
param_grid = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [None, 5, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 5, 10],
    'classifier__max_features': [None, 'sqrt', 'log2']
}

kfold = KFold(n_splits=10, shuffle=True, random_state=77)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=kfold,
    n_jobs=-1,
    scoring="recall"
)

In [25]:
with mlflow.start_run():

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    # =====================
    # MÉTRICAS
    # =====================
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_params(grid.best_params_)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    # =====================
    # MATRIZ DE CONFUSIÓN
    # =====================
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Potable", "Potable"]
    )
    disp.plot(cmap="Blues")
    plt.title("Matriz de Confusión")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()

    # =====================
    # CURVA PRECISION-RECALL
    # =====================
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall_vals, precision_vals)

    plt.figure()
    plt.plot(recall_vals, precision_vals)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Curva Precision-Recall (AUC = {pr_auc:.3f})")
    plt.savefig("precision_recall_curve.png")
    mlflow.log_artifact("precision_recall_curve.png")
    mlflow.log_metric("pr_auc", pr_auc)
    plt.close()

    # =====================
    # CURVA ROC (Muy importante también)
    # =====================
    from sklearn.metrics import roc_curve, roc_auc_score

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Curva ROC (AUC = {roc_auc:.3f})")
    plt.savefig("roc_curve.png")
    mlflow.log_artifact("roc_curve.png")
    mlflow.log_metric("roc_auc", roc_auc)
    plt.close()

    # =====================
    # LOG MODELO
    # =====================
    mlflow.sklearn.log_model(
        best_model,
        "modelo_potabilidad",
        registered_model_name="Modelo_Potabilidad"
    )

    # =====================
    # ÁRBOL
    # =====================
    plt.figure(figsize=(20,10))
    plot_tree(best_model.named_steps['classifier'], filled=True)
    plt.savefig("arbol.png")
    mlflow.log_artifact("arbol.png")
    plt.close()

2026/02/21 20:35:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 20:35:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Modelo_Potabilidad' already exists. Creating a new version of this model...
Created version '5' of model 'Modelo_Potabilidad'.


Métrica	    Antes	Ahora
Accuracy	0.64	0.45 ⬇
Recall	    0.27	0.81 
F1	        0.37	0.53 ⬆

¿Qué significa esto?

El modelo ahora:

Detecta MUCHÍSIMA más agua potable (recall 81%)

Pero comete más falsos positivos

Por eso baja el accuracy

Y aquí viene la parte importante:

En problemas desbalanceados, accuracy NO es la métrica correcta.

El modelo ahora es mucho mejor para el objetivo de negocio.


Recall = 0.81 significa:

El modelo está detectando correctamente el 81% del agua potable.

Eso es mucho mejor que antes (27%).

F1 pasó de 0.37 → 0.53
Eso es una mejora real.

Como nuestra prioridad es detectar el agua potable este modelo es el mejo